Лабораторная работа 2

В рамках данного пункта необходимо выбрать и обучить бейзлайн-модели, а также измерить их качество.



# Библиотеки

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [4]:
from sklearn.metrics import accuracy_score
from sklearn.dummy import DummyClassifier

In [5]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [6]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

In [7]:
import random

In [8]:
data = pd.read_csv('/content/drive/MyDrive/Фтми/2 курс/ОП и ПМО/student-mat.csv')

# Произведено разбиение датасета на тренировочную/тестовую выборки - 2 балла


Для начала определим целевую переменную и признаки.

Из предыдущей работы:
**Так как корреляции между итоговой оценкой G3 и другими атрибутами не самые сильные, за исключением промежуточных оценок G1 и G2, то в следубщей работе будем строить модель предсказания на основе этих оценок, а через лабораторную попробуем работать с образованием родителей, еженедельным временем обучения и качеством семейных отношений. Возможно, эти атрибуты и будут изменены.**

Таким образом, целевая переменная - G3, а призанкми G1 и G2

In [9]:
y = data['G3']
X = data[['G1', 'G2']]

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [11]:
print("Размер тренировочной выборки:", X_train.shape)
print("Размер тестовой выборки:", X_test.shape)

Размер тренировочной выборки: (276, 2)
Размер тестовой выборки: (119, 2)


# Произведено измерение качества константного предсказания (например, наиболее частотный класс для классификации, среднее/медиана для регрессии) - 3 балла


Чтобы измерить качество такого предсказания, можно использовать метрику точности (accuracy), которая показывает, какая доля предсказаний совпадает с истинными значениями.

## Контрастное предсказание

Контрастное предсказание с помощью dummy работает без входных признаков, не анализирует данные, а просто применяет заранее заданную стратегию для всех предсказаний.

In [12]:
dummy_clf = DummyClassifier(strategy='most_frequent')
dummy_clf.fit(X_train, y_train)

DummyClassifier(strategy='most_frequent')

### Предсказание на тестовой выборке и оценка качества

In [13]:
y_pred = dummy_clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print("Точность константного предсказания:", accuracy)

Точность константного предсказания: 0.058823529411764705


### Среднее значение

In [14]:
mean_prediction = np.mean(y_train)

In [15]:
y_pred = np.full(y_test.shape, mean_prediction)

In [16]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)

print("Средняя абсолютная ошибка (MAE):", mae)
print("Средняя квадратичная ошибка (MSE):", mse)

Средняя абсолютная ошибка (MAE): 3.6907197661673368
Средняя квадратичная ошибка (MSE): 22.000885609442264


***Исходя из оценки, модель работает плохо, ее нужно дорабатывать, подбирая дополнительные признаки, менять пропорцию выборок

# Бейзлайновая модель из простого семейства (линейные модели, деревья решений, knn...) обучена на тренировочной выборке, учтены особенности предобработки данных для модели, если они есть - 6 баллов


## Предобработка

проверка на пропуски

In [17]:
print(data.isnull().sum())

school        0
sex           0
age           0
address       0
famsize       0
Pstatus       0
Medu          0
Fedu          0
Mjob          0
Fjob          0
reason        0
guardian      0
traveltime    0
studytime     0
failures      0
schoolsup     0
famsup        0
paid          0
activities    0
nursery       0
higher        0
internet      0
romantic      0
famrel        0
freetime      0
goout         0
Dalc          0
Walc          0
health        0
absences      0
G1            0
G2            0
G3            0
dtype: int64


Кодирование категориальных признаков

In [18]:
data = pd.get_dummies(data, drop_first=True)

целевая перменная и признаки

In [19]:
X = data.drop('G3', axis=1) # Признаки
y = data['G3']

стандартизация данных

In [20]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

## Обучение и оценка модели

In [21]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42)

In [22]:
model = LinearRegression()
model.fit(X_train, y_train)

LinearRegression()

In [23]:
y_pred = model.predict(X_test)

# Произведено измерение качества на отложенной выборке с использованием ранее выбранной метрики - 2 балла


Выбранная метрика - Средняя абсолютная ошибка

Средняя абсолютная ошибка измеряет среднее значение абсолютных ошибок между предсказанными и фактическими значениями итоговой оценки G3. MAE является хорошей метрикой для задач регрессии, так как она дает представление о том, насколько близко предсказанные значения находятся к реальным значениям. Эта метрика выбрана, так как она обеспечивает устойчивость к выбросам, поскольку все ошибки взвешиваются одинаково. Также легко сравнивать производительность разных моделей.

In [24]:
mae = mean_absolute_error(y_test, y_pred)

print("Средняя абсолютная ошибка (MAE):", mae)

Средняя абсолютная ошибка (MAE): 1.5250573393887548


*Действитнльо, с изменением признаков ошибки с 3,6 баллов снизилась в 2 раза*

Учитывая, что диапазон оценок от 1 до 20, то ошибку 1,5 для простой модели можно считать небольшой